In [1]:
"""
EVisionary — Comprehensive Validation Suite (key_B)
Single-file, reproducible. Each SECTION writes a supplementary-ready table.
"""

import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from evisionary_common import (
    QUERY_PATTERNS, valid_text_series, contains_ci,
    count_unique_pmids, SOURCE_DISPLAY_ORDER,
)

# ---------------------------------------------------------------
# PATHS (self-contained so the script never inherits a stale path)
# ---------------------------------------------------------------
KEYB_OUTPUT_DIR = "/Users/sogand/EVisionary_outputs_keyB"
Path(KEYB_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

DATA_PATH_MASTER = os.path.join(KEYB_OUTPUT_DIR, "unified_EVmetadata_keyB.parquet")
DATA_PATH_ENRICHED = os.path.join(KEYB_OUTPUT_DIR, "unified_EVmetadata_enriched.parquet")

OUT_DIR = Path("/Users/sogand/Desktop/Article_Figures/keyB")
AUDIT_DIR = OUT_DIR / "Validation_Audits"
OUT_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

# Plot styling
plt.rcParams["font.family"] = ["Arial", "sans-serif"]
C_BG, C_NAVY = "#F8FAFB", "#1f4e79"
LINE_COLORS = {"species": "#1f4e79", "sample_name": "#4682B4",
               "disease": "#2E8B57", "method": "#7A9E9F", "molecule_type": "#8B5E83"}
MIN_RECORDS_PER_YEAR, ROLLING_WINDOW, YEAR_MIN, YEAR_MAX = 50, 3, 2005, 2023

MIR_PATTERN = r"mir|micro\s*rna"


def header(title):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)


# ===============================================================
# SECTION 1: METHOD FIELD AUDIT
# ===============================================================
def section_method_audit(df):
    header("SECTION 1: METHOD FIELD AUDIT")
    method = valid_text_series(df["method"])
    missing_mask = method.eq("")
    print(f"Missing/unusable method: {missing_mask.sum():,} ({missing_mask.mean()*100:.2f}%)")
    vc = (method.replace("", "[missing]")
          .value_counts().rename_axis("Method label").reset_index(name="Count"))
    vc.to_csv(OUT_DIR / "TableS3_Method_ValueCounts.csv", index=False)
    print(vc.head(15).to_string(index=False))


# ===============================================================
# SECTION 2: SYNTAX SENSITIVITY
# ===============================================================
def section_syntax_sensitivity(df):
    header("SECTION 2: SYNTAX SENSITIVITY")
    rows = []
    scenarios = [
        ("molecule_type", "Protein family", ["protein", "Protein", "PROTEIN"]),
        ("molecule_type", "miRNA family", ["mirna", "miRNA", "miRNAs"]),
        ("disease", "Breast cancer family", ["breast cancer", "Breast Cancer", "breast carcinoma"]),
    ]
    for field, family, variants in scenarios:
        s = valid_text_series(df[field])
        for term in variants:
            exact = int((s == term).sum())
            robust = int(contains_ci(s, re.escape(term)).sum())
            rows.append({
                "Field": field, "Concept Family": family, "Input Variant": term,
                "Strict Exact": exact, "Robust CI": robust,
                "Absolute Gain": robust - exact,
                "Fold Change": round(robust / exact, 4) if exact > 0 else np.nan,
            })
    out = pd.DataFrame(rows)
    out.to_csv(OUT_DIR / "TableS4_Syntax_Sensitivity.csv", index=False)
    print(out.to_string(index=False, na_rep="NA"))


# ===============================================================
# SECTION 3: TIME-STRATIFIED COMPLETENESS
# ===============================================================
def section_time_completeness(df):
    header("SECTION 3: TIME-STRATIFIED COMPLETENESS")
    df2 = df.copy()
    df2["year"] = pd.to_numeric(df2["year"], errors="coerce")
    df2 = df2[df2["year"].notna()].copy()
    df2["year"] = df2["year"].astype(int)
    df2 = df2[(df2["year"] >= YEAR_MIN) & (df2["year"] <= YEAR_MAX)].copy()

    fields = [c for c in ["species", "sample_name", "disease", "method", "molecule_type"]
              if c in df2.columns]
    for col in fields:
        df2[f"{col}_present"] = (valid_text_series(df2[col]) != "")
    presence_cols = [f"{c}_present" for c in fields]

    year_counts = df2.groupby("year").size().rename("Records")
    summary = (df2.groupby("year")[presence_cols].mean() * 100)
    summary.columns = fields
    summary = summary.join(year_counts)
    summary = summary[summary["Records"] >= MIN_RECORDS_PER_YEAR]
    summary.reset_index().to_csv(OUT_DIR / "TableS6_TimeCompleteness.csv", index=False)
    print(summary[fields].round(2).to_string())

    plot_df = summary[fields].rolling(window=ROLLING_WINDOW, min_periods=1).mean()
    fig, ax = plt.subplots(figsize=(12, 6.2), facecolor=C_BG)
    ax.set_facecolor(C_BG)
    for col in fields:
        ax.plot(plot_df.index, plot_df[col], marker="o", markersize=5,
                linewidth=2.4, label=col, color=LINE_COLORS.get(col, C_NAVY))
    for sp in ["top", "right"]:
        ax.spines[sp].set_visible(False)
    ax.set_ylim(0, 100)
    ax.set_title("Time-stratified metadata completeness", color=C_NAVY,
                 fontsize=16, fontweight="bold")
    ax.set_xlabel("Publication year", color=C_NAVY, fontweight="bold")
    ax.set_ylabel("Completeness (%)", color=C_NAVY, fontweight="bold")
    ax.legend(frameon=False, bbox_to_anchor=(1.02, 1.0), loc="upper left")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "FigS3_Time_Completeness.png", dpi=600,
                facecolor=C_BG, bbox_inches="tight")
    plt.savefig(OUT_DIR / "FigS3_Time_Completeness.pdf",
                facecolor=C_BG, bbox_inches="tight")
    plt.close()


# ===============================================================
# SECTION 4: miRNA PRESENCE & ROUTING DIAGNOSTIC (reversed)
# After multi-cargo integration, miRNA is a populated class. This
# section documents its recovery rather than its absence.
# ===============================================================
def section_mirna_diagnostic(df):
    header("SECTION 4: miRNA PRESENCE & ROUTING DIAGNOSTIC")

    mir_summary = []
    for col in ["molecule_type_raw", "molecule_type_norm", "molecule_type_group", "molecule_type"]:
        if col in df.columns:
            s = df[col].fillna("").astype(str)
            mask = s.str.contains(MIR_PATTERN, case=False, regex=True, na=False)
            mir_summary.append([col, int(mask.sum()), int(s[mask].nunique())])
            if mask.sum() > 0:
                print(f"\n{col}: {int(mask.sum()):,} matches")
                print(s[mask].value_counts().head(10).to_string())
    pd.DataFrame(mir_summary, columns=["Column", "Matches", "Unique labels"]).to_csv(
        OUT_DIR / "TableS5_miRNA_Diagnostic.csv", index=False)

    if "molecule_type_raw" in df.columns:
        raw = df["molecule_type_raw"].fillna("").astype(str)
        mir_mask = raw.str.contains(MIR_PATTERN, case=False, regex=True, na=False)
        if mir_mask.any():
            routing = (df.loc[mir_mask, ["molecule_type_raw", "molecule_type_norm",
                                         "molecule_type_group"]]
                       .value_counts().reset_index(name="count"))
            routing.to_csv(OUT_DIR / "TableS5A_miRNA_Routing.csv", index=False)
            n_mir = int((valid_text_series(df["molecule_type"]).str.casefold() == "mirna").sum())
            print(f"\nmiRNA is PRESENT: {n_mir:,} canonical miRNA records "
                  f"(recovered via multi-cargo integration across Vesiclepedia + ExoCarta).")
            print("\nROUTING (raw miR-like -> canonical):")
            print(routing.to_string(index=False))
        else:
            print("\nNo miR-like labels found in molecule_type_raw.")


# ===============================================================
# SECTION 5: SOURCE COMPLEMENTARITY (PMID-level)
# ===============================================================
def _ordered_sources(vals):
    vals = set(vals)
    return ([s for s in SOURCE_DISPLAY_ORDER if s in vals]
            + sorted(vals - set(SOURCE_DISPLAY_ORDER)))


def _comp_queries(df):
    return {
        "Protein": contains_ci(valid_text_series(df["molecule_type"]), QUERY_PATTERNS["Protein"]),
        "miRNA": contains_ci(valid_text_series(df["molecule_type"]), QUERY_PATTERNS["miRNA"]),
        "Lipid": contains_ci(valid_text_series(df["molecule_type"]), QUERY_PATTERNS["Lipid"]),
        "Plasma": contains_ci(valid_text_series(df["sample_name"]), QUERY_PATTERNS["Plasma"]),
        "BreastCancer": contains_ci(valid_text_series(df["disease"]), QUERY_PATTERNS["Breast Cancer"]),
    }


def section_source_complementarity(df):
    header("SECTION 5: SOURCE COMPLEMENTARITY")
    comp_rows = []
    for qname, mask in _comp_queries(df).items():
        sub = df.loc[mask, ["pmid", "source"]].copy()
        sub["pmid"] = valid_text_series(sub["pmid"])
        sub = sub[sub["pmid"] != ""].drop_duplicates()
        if sub.empty:
            continue
        pmid_src = sub.groupby("pmid")["source"].apply(_ordered_sources).reset_index()
        pmid_src["Combination"] = pmid_src["source"].apply(lambda x: " | ".join(x))
        pc = pmid_src["Combination"].value_counts().reset_index()
        pc.columns = ["Source combination", "Unique PMIDs"]
        pc["Scenario"] = qname
        comp_rows.append(pc)
    if comp_rows:
        out = pd.concat(comp_rows, ignore_index=True)
        out.to_csv(OUT_DIR / "TableS8_Source_Complementarity.csv", index=False)
        print(out.to_string(index=False))


# ===============================================================
# SECTION 6: RECORD-TO-PMID DENSITY
# ===============================================================
def section_density(df):
    header("SECTION 6: RECORD-TO-PMID DENSITY")
    rows = []
    for scenario, mask in _comp_queries(df).items():
        n = int(mask.sum())
        p = count_unique_pmids(df, mask)
        rows.append([scenario, n, p, round(n / p, 2) if p > 0 else np.nan])
    out = pd.DataFrame(rows, columns=["Scenario", "Records", "Unique PMIDs", "Records/PMID"])
    out.to_csv(OUT_DIR / "TableS9_Density.csv", index=False)
    print(out.to_string(index=False))


# ===============================================================
# SECTION 7: ABLATION (two-stage molecule + one-stage stabilization)
# ===============================================================
def section_ablation(df):
    header("SECTION 7: ABLATION")

    # TWO-STAGE: molecule_type classes (raw -> stable -> harmonized)
    two_stage = []
    for concept, exact_val, pattern in [
        ("Protein", "protein", QUERY_PATTERNS["Protein"]),
        ("mRNA", "mrna", QUERY_PATTERNS["mRNA"]),
        ("miRNA", "mirna", QUERY_PATTERNS["miRNA"]),
        ("Lipid", "lipid", QUERY_PATTERNS["Lipid"]),
    ]:
        raw_s = valid_text_series(df["molecule_type_raw"])
        canon_s = valid_text_series(df["molecule_type"])
        m_exact = raw_s.str.casefold().eq(exact_val)
        m_stable = contains_ci(raw_s, pattern)
        m_harm = contains_ci(canon_s, pattern)
        c_e, c_s, c_h = int(m_exact.sum()), int(m_stable.sum()), int(m_harm.sum())
        if c_h == 0 and c_e == 0:
            print(f"[skip] {concept} absent.")
            continue
        two_stage.append({
            "Concept": concept, "Analysis": "Two-stage (molecule_type)",
            "Raw_Exact": c_e, "Raw_Stable": c_s, "Canonical_Harmonized": c_h,
            "Gain_Exact_to_Stable": c_s - c_e, "Gain_Stable_to_Harmonized": c_h - c_s,
            "Total_Gain": c_h - c_e,
            "Fold": round(c_h / c_e, 2) if c_e > 0 else None,
            "Unique_PMIDs": count_unique_pmids(df, m_harm),
        })

    # ONE-STAGE: stabilization only (species / sample / disease)
    one_stage = []
    for concept, field, exact_val in [
        ("Homo sapiens", "species", "homo sapiens"),
        ("Rattus norvegicus", "species", "rattus norvegicus"),
        ("Plasma", "sample_name", "plasma"),
        ("Breast Cancer", "disease", "breast cancer"),
    ]:
        s = valid_text_series(df[field])
        pattern = QUERY_PATTERNS.get(concept, rf"\b{exact_val}\b")
        m_exact = s.str.casefold().eq(exact_val)
        m_stable = contains_ci(s, pattern)
        c_e, c_s = int(m_exact.sum()), int(m_stable.sum())
        one_stage.append({
            "Concept": concept, "Analysis": "One-stage (stabilization only)",
            "Field": field, "Exact": c_e, "Stabilized": c_s,
            "Stabilization_Gain": c_s - c_e,
            "Fold": round(c_s / c_e, 2) if c_e > 0 else None,
            "Unique_PMIDs": count_unique_pmids(df, m_stable),
            "Note": "No separate raw column; only syntax stabilization measurable.",
        })

    # Composite: Homo sapiens + Protein
    hs = valid_text_series(df["species"])
    prot = valid_text_series(df["molecule_type"])
    m_comp_exact = hs.str.casefold().eq("homo sapiens") & prot.str.casefold().eq("protein")
    m_comp_stable = (contains_ci(hs, QUERY_PATTERNS["Homo sapiens"])
                     & contains_ci(prot, QUERY_PATTERNS["Protein"]))
    one_stage.append({
        "Concept": "Homo sapiens + Protein", "Analysis": "Composite (stabilization)",
        "Field": "species+molecule_type",
        "Exact": int(m_comp_exact.sum()), "Stabilized": int(m_comp_stable.sum()),
        "Stabilization_Gain": int(m_comp_stable.sum() - m_comp_exact.sum()),
        "Fold": round(m_comp_stable.sum() / m_comp_exact.sum(), 2) if m_comp_exact.sum() > 0 else None,
        "Unique_PMIDs": count_unique_pmids(df, m_comp_stable),
        "Note": "Composite stabilization across two fields.",
    })

    pd.DataFrame(two_stage).to_csv(AUDIT_DIR / "Table_Ablation_TwoStage.csv", index=False)
    pd.DataFrame(one_stage).to_csv(AUDIT_DIR / "Table_Ablation_OneStage.csv", index=False)
    print("Two-stage:\n", pd.DataFrame(two_stage).to_string(index=False))
    print("\nOne-stage:\n", pd.DataFrame(one_stage).to_string(index=False))


# ===============================================================
# SECTION 8: ERROR TAXONOMY (static, evidence-anchored)
# ===============================================================
def section_error_taxonomy():
    header("SECTION 8: ERROR TAXONOMY")
    taxonomy = [
        {"Failure_Mode": "Missing contextual metadata",
         "Operational_Definition": "Absence/sparsity of key contextual variables despite record presence",
         "Representative_Evidence": "Field-level completeness audit (FigS3; source-stratified tables)",
         "Practical_Consequence": "Clinical/phenotypic queries return few or no records"},
        {"Failure_Mode": "Label heterogeneity and syntax fragility",
         "Operational_Definition": "Case/format variation in otherwise equivalent labels",
         "Representative_Evidence": "Ablation and syntax-sensitivity audit (TableS4)",
         "Practical_Consequence": "Strict exact matching under-recovers valid records"},
        {"Failure_Mode": "Cross-field semantic leakage",
         "Operational_Definition": "Entities appearing outside expected canonical fields",
         "Representative_Evidence": "miR-like routing diagnostic (TableS5)",
         "Practical_Consequence": "Class-based retrieval depends on correct field routing"},
        {"Failure_Mode": "Semantic field misalignment",
         "Operational_Definition": "Populated fields dominated by mismatched content types",
         "Representative_Evidence": "Method-field raw-label audit (TableS3)",
         "Practical_Consequence": "Nominal completeness overestimates interpretability"},
        {"Failure_Mode": "Source-specific coverage asymmetry",
         "Operational_Definition": "Annotations concentrated in a single source",
         "Representative_Evidence": "Source complementarity (TableS8) and density analyses",
         "Practical_Consequence": "Retrieval depends strongly on repository composition"},
    ]
    pd.DataFrame(taxonomy).to_csv(AUDIT_DIR / "Table_Error_Taxonomy.csv", index=False)
    print(f"Taxonomy rows: {len(taxonomy)}")


# ===============================================================
# SECTION 9: SOURCE-BALANCED MANUAL SPOT-CHECK SAMPLES
# ===============================================================
def _source_balanced_sample(df_sub, n_target=15, seed=42):
    df_sub = df_sub.copy()
    df_sub["_src"] = valid_text_series(df_sub["source"])
    df_sub = df_sub[df_sub["_src"] != ""]
    if df_sub.empty:
        return df_sub.drop(columns=["_src"])
    if len(df_sub) <= n_target:
        return df_sub.sample(frac=1, random_state=seed).drop(columns=["_src"])
    sources = df_sub["_src"].unique()
    n_per = max(1, n_target // len(sources))
    parts = [df_sub[df_sub["_src"] == s].sample(
                 n=min(len(df_sub[df_sub["_src"] == s]), n_per), random_state=seed)
             for s in sources]
    out = pd.concat(parts)
    short = n_target - len(out)
    if short > 0:
        rem = df_sub.drop(out.index)
        if not rem.empty:
            out = pd.concat([out, rem.sample(n=min(len(rem), short), random_state=seed)])
    return out.sample(frac=1, random_state=seed).drop(columns=["_src"])


def section_spot_check(df):
    header("SECTION 9: MANUAL SPOT-CHECK SAMPLES")
    base_cols = ["pmid", "source", "species", "sample_name", "disease",
                 "molecule_type", "method", "working_id"]
    cols_present = [c for c in base_cols if c in df.columns]

    spot_queries = [
        ("Plasma", "sample_name",
         contains_ci(valid_text_series(df["sample_name"]), QUERY_PATTERNS["Plasma"])),
        ("Breast Cancer", "disease",
         contains_ci(valid_text_series(df["disease"]), QUERY_PATTERNS["Breast Cancer"])),
        ("Protein", "molecule_type",
         contains_ci(valid_text_series(df["molecule_type"]), QUERY_PATTERNS["Protein"])),
        ("miRNA", "molecule_type",
         contains_ci(valid_text_series(df["molecule_type"]), QUERY_PATTERNS["miRNA"])),
        ("Lipid", "molecule_type",
         contains_ci(valid_text_series(df["molecule_type"]), QUERY_PATTERNS["Lipid"])),
    ]
    samples = []
    for concept, field, mask in spot_queries:
        sub = df[mask]
        print(f"{concept}: {len(sub):,} matched, {count_unique_pmids(df, mask)} PMIDs")
        if len(sub) > 0:
            s = _source_balanced_sample(sub, 15)[cols_present].copy()
            s.insert(0, "Target_Query", concept)
            s.insert(1, "Matched_Field", field)
            s["Is_Valid_Match_Y_N"] = ""
            s["Reviewer_Notes"] = ""
            samples.append(s)
    if samples:
        out = pd.concat(samples, ignore_index=True)
        out.to_csv(AUDIT_DIR / "Table_Manual_SpotCheck.csv", index=False)
        print(f"Spot-check rows: {len(out)}")


# ===============================================================
# MAIN
# ===============================================================
def main():
    header("EVisionary — Comprehensive Validation Suite (key_B)")
    df = pd.read_parquet(DATA_PATH_MASTER)
    print(f"Loaded {len(df):,} rows from {DATA_PATH_MASTER}")

    section_method_audit(df)
    section_syntax_sensitivity(df)
    section_time_completeness(df)
    section_mirna_diagnostic(df)
    section_source_complementarity(df)
    section_density(df)
    section_ablation(df)
    section_error_taxonomy()
    section_spot_check(df)

    header("VALIDATION SUITE COMPLETE")
    print(f"Tables/figures written to: {OUT_DIR}")
    print(f"Audit tables written to:   {AUDIT_DIR}")


if __name__ == "__main__":
    main()


EVisionary — Comprehensive Validation Suite (key_B)
Loaded 258,460 rows from /Users/sogand/EVisionary_outputs_keyB/unified_EVmetadata_keyB.parquet

SECTION 1: METHOD FIELD AUDIT
Missing/unusable method: 43,597 (16.87%)
                                                                                                                   Method label  Count
                                                                                                                      [missing]  43597
                                                                                Differential centrifugation|Ultracentrifugation  41667
                                                                     Differential centrifugation|Filtration|Ultracentrifugation  38627
                                                                                                    Differential centrifugation  11346
                                                          Differential centrifugation|Ultrafiltration|Opt